# Train HeadScan-Lite in Google Colab

Use a GPU runtime and choose **Runtime → Run all**. This trains the Colab/mobile-oriented variable-view 3D head reconstruction model from this repository.

The notebook follows the same bootstrap pattern as `resistor_model`: it makes a fresh shallow clone of `main`, prints an explicit `HeadScan-Lite Colab version:` line, mounts Drive, installs the repo, then launches the versioned Python trainer with visible heartbeat/progress output.

### Real dataset

Place `headscan_dataset.zip` at `MyDrive/head_model/headscan_dataset.zip`. The archive must contain `train.jsonl` and `valid.jsonl` plus the referenced registered meshes/images. See `docs/dataset-format.md`.

All meshes must have identical registered topology. Vertices and depth should preferably be in millimetres. RGB is required; normal/depth/mask supervision is strongly recommended.

If no real dataset is present, the recipe creates a **tiny procedural smoke-test dataset** and trains only two epochs. This is solely to verify setup, training, checkpoint/resume and ONNX export; it does **not** create a useful real head-scanning model.

Persistent real-training outputs go to `MyDrive/head_model/headscan-lite`. `last.pt` is saved every epoch, `best.pt` whenever validation mean vertex error improves, numbered checkpoints every 5 epochs, and the PCA shape basis + metrics remain in Drive. A Drive `last.pt` is staged to local `/content` with visible copy progress before resume.

Default real training: 256×256, 1–8 randomly sampled views per identity, batch size 2, up to 128 PCA components, MobileNetV3-Small, 3-layer/4-head variable-view Transformer, AMP, and geometry + normal/depth/mask/confidence losses.

After training, the geometry-only path is exported to ONNX. The mobile export returns metric mesh vertices, normalized shape coefficients, and per-view quality weights while skipping the dense training-only heads.

In [ ]:
import pathlib, shutil, subprocess
bootstrap = pathlib.Path('/content/head_model_bootstrap')
if bootstrap.exists():
    shutil.rmtree(bootstrap)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Persie0/Head_model.git', str(bootstrap)], check=True)
subprocess.run(['git', '-C', str(bootstrap), 'rev-parse', '--short', 'HEAD'], check=True)
script = bootstrap / 'colab' / 'train_headscan_lite_colab.py'
source = script.read_text(encoding='utf-8')
required = ['HEADSCAN_COLAB_VERSION', 'PROGRESS_EVERY = 1', 'run_with_heartbeat', 'ALLOW_SYNTHETIC_SMOKE_TEST']
if not all(token in source for token in required):
    raise RuntimeError('Fresh clone does not contain the expected current HeadScan-Lite Colab trainer. Stop instead of running stale code.')
exec(compile(source, str(script), 'exec'), {'__name__': '__main__'})
